[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Charles9257/PhD-Datasets-Analysis/blob/claude/project-thread-9pn62q/notebooks/SER_Privacy_Dataset_Prep.ipynb)

# SER privacy: dataset preparation, exploration and first leakage checks

This notebook supports the PhD work on **paralinguistic inference attacks in federated speech emotion recognition**. It:

1. Loads IEMOCAP, CREMA-D, RAVDESS, EmoDB and SAVEE into **one metadata table** (file, corpus, emotion, speaker, gender, age).
2. Explores the data: class and demographic distributions, waveforms, spectrograms, audio playback.
3. Preprocesses audio: 16 kHz resampling, silence trimming, loudness normalisation, MFCC and log-mel features, wav2vec2 / HuBERT embeddings.
4. Runs quick model tests: an SER baseline (UAR, macro-F1), gender / speaker / age **attribute probes** on the embeddings (first leakage check), and a minimal **FedAvg** simulation with speaker-partitioned clients, including a server-side gender inference attack on client updates and an optional DP (clip + Gaussian noise) switch.

**Source data is never modified.** Everything the notebook writes goes to `OUT_DIR` (on Colab: `MyDrive/SER_outputs`, so results survive the session; locally: an `outputs` folder next to this notebook).

Laptop-friendly: set `N_PER_CORPUS` (utterances sampled per corpus) and `MAX_SECONDS` in the config cell. Start small (e.g. 200) and scale up.

**On Google Colab:** Runtime > Change runtime type > **T4 GPU**, then run cells top to bottom. The notebook mounts your Google Drive and reads the datasets from `DATA_ROOT` (default `/content/drive/MyDrive/Datasets`), which should mirror your local `C:\Desktop\RGU PhD\New folder\Datasets` layout. Works locally too: the Colab-only steps are skipped.

## 0. Environment setup (Colab or local)

Detects Colab, mounts Google Drive, installs the few packages Colab does not ship with, and checks for a GPU. On Colab, choose **Runtime > Change runtime type > T4 GPU** before running; the SSL embedding step is much faster on GPU.

Locally, Python 3.10 or 3.11 is recommended; install `requirements.txt` once instead (`pip install -r requirements.txt`).

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Colab already has numpy, pandas, matplotlib, seaborn, librosa, soundfile, scikit-learn, tqdm, torch, transformers.
    # This just makes sure they are present and recent enough.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "librosa>=0.10", "soundfile", "transformers>=4.40", "ipywidgets"], check=True)
else:
    print("Not on Colab: skipping Drive mount. Install dependencies once with: pip install -r requirements.txt")

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("No GPU found, running on CPU. On Colab: Runtime > Change runtime type > T4 GPU, then re-run from the top.")


## 1. Configuration

In [ ]:
from pathlib import Path
import os, re, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---- Paths (edit if you move things) ----
# DATA_ROOT mirrors the local folder C:\Desktop\RGU PhD\New folder\Datasets.
# On Colab, upload (or sync) that folder to Google Drive as MyDrive/Datasets, or change the path below.
# You can also override it with an environment variable: os.environ["DATA_ROOT"] = "/content/drive/MyDrive/..."
IN_COLAB = "google.colab" in __import__("sys").modules
DEFAULT_ROOT = "/content/drive/MyDrive/Datasets" if IN_COLAB else r"C:\Desktop\RGU PhD\New folder\Datasets"
DATA_ROOT = Path(os.environ.get("DATA_ROOT", DEFAULT_ROOT))
PATHS = {
    "IEMOCAP_DIR": DATA_ROOT / "IEMOCAP_full_release",
    "IEMOCAP_CSV": DATA_ROOT / "IEMOCAP_full_release" / "iemocap_full_dataset.csv",
    "CREMA_WAV":   DATA_ROOT / "Datasets" / "Crema",                       # complete 7,442 wav (the AudioMP3 folder is partial)
    "CREMA_DEMO":  DATA_ROOT / "CREMA-D-master" / "VideoDemographics.csv",
    "RAVDESS_WAV": DATA_ROOT / "Datasets" / "Ravdess" / "audio_speech_actors_01-24",
    "EMODB_WAV":   DATA_ROOT / "Datasets" / "EmoDB",
    "EMODB_DEMO":  DATA_ROOT / "Datasets" / "emoDB demographics - Sheet1.csv",
    "SAVEE_WAV":   DATA_ROOT / "Datasets" / "Savee",
}
CORPORA = ["IEMOCAP", "CREMA-D", "RAVDESS", "EmoDB", "SAVEE"]   # drop any you don't want

# ---- Outputs (never inside the source folders) ----
if IN_COLAB:
    OUT_DIR = Path("/content/drive/MyDrive/SER_outputs")   # on Drive so results and cached features survive the session
else:
    try:
        NB_DIR = Path(__vsc_ipynb_file__).parent          # VS Code
    except NameError:
        NB_DIR = Path.cwd()                               # Jupyter: folder the notebook was opened from
    OUT_DIR = NB_DIR / "outputs"
(OUT_DIR / "features").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

# ---- Laptop knobs ----
SEED = 42
N_PER_CORPUS = 300          # utterances sampled per corpus for features/models (None = all; slow)
MAX_SECONDS = 6.0           # crop long utterances for embedding extraction
TARGET_SR = 16_000
IEMOCAP_4CLASS = True       # standard setup: ang / hap(+exc) / neu / sad
SSL_MODELS = {"wav2vec2": "facebook/wav2vec2-base", "hubert": "facebook/hubert-base-ls960"}

random.seed(SEED); np.random.seed(SEED)
for k, p in PATHS.items():
    print(f"{k:12s} {'OK ' if p.exists() else 'MISSING'}  {p}")
print("Outputs ->", OUT_DIR)
if not DATA_ROOT.exists():
    print(f"\nDATA_ROOT not found: {DATA_ROOT}\nCheck the Drive folder name, or set DATA_ROOT above.")
missing = [c for c, keys in {"IEMOCAP": ["IEMOCAP_CSV"], "CREMA-D": ["CREMA_WAV", "CREMA_DEMO"], "RAVDESS": ["RAVDESS_WAV"],
                             "EmoDB": ["EMODB_WAV", "EMODB_DEMO"], "SAVEE": ["SAVEE_WAV"]}.items()
           if c in CORPORA and not all(PATHS[k].exists() for k in keys)]
if missing:
    print("Dropping corpora with missing files:", missing)
    CORPORA = [c for c in CORPORA if c not in missing]

## 2. Unified metadata table

One row per utterance with columns `path, corpus, emotion, emotion_raw, speaker, gender, age, extra`. Speaker IDs are prefixed with the corpus so they are globally unique. Emotions are mapped to a shared vocabulary (`neutral, happy, sad, angry, fear, disgust, surprise, calm, boredom, frustrated, other`).

In [ ]:
def load_iemocap():
    df = pd.read_csv(PATHS["IEMOCAP_CSV"])
    m = {"neu":"neutral","hap":"happy","exc":"happy" if IEMOCAP_4CLASS else "excited","sad":"sad","ang":"angry",
         "fru":"frustrated","fea":"fear","sur":"surprise","dis":"disgust","oth":"other","xxx":None}
    rows = []
    for r in df.itertuples():
        rel = str(r.path).replace("\\", "/")         # CSV may hold Windows-style paths
        stem = Path(rel).stem                     # e.g. Ses01F_script02_1_F000
        spk_sex = stem.split("_")[-1][0]             # F/M of the actual speaker
        rows.append(dict(path=str(PATHS["IEMOCAP_DIR"] / rel), corpus="IEMOCAP",
                         emotion=m.get(r.emotion), emotion_raw=r.emotion,
                         speaker=f"IEM_{stem[:5]}{spk_sex}", gender="female" if spk_sex=="F" else "male",
                         age=np.nan, extra=f"session={r.session};method={r.method};agree={r.agreement}"))
    out = pd.DataFrame(rows).dropna(subset=["emotion"])
    if IEMOCAP_4CLASS:
        out = out[out.emotion.isin(["angry","happy","neutral","sad"])]
    return out

def load_crema():
    demo = pd.read_csv(PATHS["CREMA_DEMO"]).set_index("ActorID")
    m = {"ANG":"angry","DIS":"disgust","FEA":"fear","HAP":"happy","NEU":"neutral","SAD":"sad"}
    rows = []
    for f in sorted(PATHS["CREMA_WAV"].glob("*.wav")):
        a, sent, emo, lvl = f.stem.split("_")
        d = demo.loc[int(a)]
        rows.append(dict(path=str(f), corpus="CREMA-D", emotion=m[emo], emotion_raw=emo, speaker=f"CRE_{a}",
                         gender=d.Sex.lower(), age=float(d.Age), extra=f"sentence={sent};level={lvl};race={d.Race}"))
    return pd.DataFrame(rows)

def load_ravdess():
    m = {"01":"neutral","02":"calm","03":"happy","04":"sad","05":"angry","06":"fear","07":"disgust","08":"surprise"}
    rows = []
    for f in sorted(PATHS["RAVDESS_WAV"].rglob("*.wav")):
        mod, voc, emo, inten, stmt, rep, actor = f.stem.split("-")
        rows.append(dict(path=str(f), corpus="RAVDESS", emotion=m[emo], emotion_raw=emo, speaker=f"RAV_{actor}",
                         gender="male" if int(actor) % 2 else "female", age=np.nan,
                         extra=f"intensity={inten};statement={stmt};rep={rep}"))
    return pd.DataFrame(rows)

def load_emodb():
    demo = pd.read_csv(PATHS["EMODB_DEMO"]).set_index("ActorID")
    m = {"W":"angry","L":"boredom","E":"disgust","A":"fear","F":"happy","T":"sad","N":"neutral"}
    rows = []
    for f in sorted(PATHS["EMODB_WAV"].glob("*.wav")):
        spk, emo = int(f.stem[:2]), f.stem[5]
        d = demo.loc[spk]
        rows.append(dict(path=str(f), corpus="EmoDB", emotion=m[emo], emotion_raw=emo, speaker=f"EMO_{spk:02d}",
                         gender=d.Gender.lower(), age=float(d.Age), extra=f"text={f.stem[2:5]}"))
    return pd.DataFrame(rows)

def load_savee():
    m = {"a":"angry","d":"disgust","f":"fear","h":"happy","n":"neutral","sa":"sad","su":"surprise"}
    rows = []
    for f in sorted(PATHS["SAVEE_WAV"].glob("*.wav")):
        spk, code_ = f.stem.split("_")
        emo = re.match(r"[a-z]+", code_).group()
        rows.append(dict(path=str(f), corpus="SAVEE", emotion=m[emo], emotion_raw=emo, speaker=f"SAV_{spk}",
                         gender="male", age=np.nan, extra=""))
    return pd.DataFrame(rows)

LOADERS = {"IEMOCAP": load_iemocap, "CREMA-D": load_crema, "RAVDESS": load_ravdess, "EmoDB": load_emodb, "SAVEE": load_savee}
meta = pd.concat([LOADERS[c]() for c in CORPORA], ignore_index=True)
meta.to_csv(OUT_DIR / "unified_metadata.csv", index=False)
print(meta.shape); meta.sample(5, random_state=SEED)

In [ ]:
summary = meta.groupby("corpus").agg(utterances=("path","size"), speakers=("speaker","nunique"),
                                     female_spk=("speaker", lambda s: meta.loc[s.index].query("gender=='female'").speaker.nunique()),
                                     has_age=("age", lambda a: a.notna().mean()), emotions=("emotion","nunique"))
summary

## 3. Exploration and visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
pd.crosstab(meta.corpus, meta.emotion).plot.bar(stacked=True, ax=axes[0], colormap="tab20"); axes[0].set_title("Emotion classes per corpus")
spk = meta.drop_duplicates("speaker")
pd.crosstab(spk.corpus, spk.gender).plot.bar(ax=axes[1]); axes[1].set_title("Speakers by gender")
for c, g in meta.drop_duplicates("speaker").dropna(subset=["age"]).groupby("corpus"):
    axes[2].hist(g.age, bins=15, alpha=.6, label=c)
axes[2].legend(); axes[2].set_title("Speaker age (only CREMA-D, EmoDB have age)")
for a in axes[:2]: a.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / "distributions.png", dpi=150); plt.show()

# Emotion x gender balance matters for attribute leakage: emotion and gender are often correlated in acted corpora
pd.crosstab([meta.corpus, meta.emotion], meta.gender, normalize="index").round(2).unstack(0)["female"]

In [ ]:
import librosa, librosa.display, soundfile as sf
from IPython.display import Audio, display

def load_audio(path, sr=TARGET_SR, max_s=None):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_s)
    return y

def show_example(row):
    y = load_audio(row.path)
    fig, ax = plt.subplots(1, 2, figsize=(14, 3))
    librosa.display.waveshow(y, sr=TARGET_SR, ax=ax[0]); ax[0].set_title(f"{row.corpus} | {row.emotion} | {row.speaker} | {row.gender}")
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_mels=64), ref=np.max)
    librosa.display.specshow(S, sr=TARGET_SR, x_axis="time", y_axis="mel", ax=ax[1]); ax[1].set_title("log-mel")
    plt.show(); display(Audio(y, rate=TARGET_SR))

for _, row in meta.groupby("corpus").sample(1, random_state=SEED).iterrows():
    show_example(row)

In [ ]:
# Duration statistics on a sample (reading headers only, fast)
def duration(p):
    try: return sf.info(p).duration
    except Exception: return librosa.get_duration(path=p)
samp = pd.concat([g.sample(min(len(g), 200), random_state=SEED) for _, g in meta.groupby("corpus")])
samp["dur"] = samp.path.map(duration)
samp.groupby("corpus").dur.describe().round(2)

## 4. Preprocessing and features

Pipeline per utterance: load mono, resample to 16 kHz, trim leading/trailing silence (`top_db=30`), peak-normalise, crop to `MAX_SECONDS`. Then:
* **MFCC** (40 coefficients + deltas, mean and std pooled → 160-d)
* **log-mel** (64 bands, mean and std pooled → 128-d)
* **SSL embeddings**: wav2vec2-base and HuBERT-base, mean-pooled per layer (so you can see which layers leak most).

Features are cached in `outputs/features/`.

In [ ]:
def sample_meta(meta, n=N_PER_CORPUS):
    # sample utterances per corpus, spreading across speakers
    if n is None: return meta.copy()
    out = []
    for c, g in meta.groupby("corpus"):
        per = max(1, n // g.speaker.nunique())
        s = pd.concat([x.sample(min(len(x), per), random_state=SEED) for _, x in g.groupby("speaker")])
        if len(s) < n: s = pd.concat([s, g.drop(s.index).sample(min(n - len(s), len(g) - len(s)), random_state=SEED)])
        out.append(s.head(n))
    return pd.concat(out).reset_index(drop=True)

work = sample_meta(meta)
print(work.groupby("corpus").agg(n=("path","size"), speakers=("speaker","nunique")))

def preprocess(path):
    y = load_audio(path)
    y, _ = librosa.effects.trim(y, top_db=30)
    y = y / (np.abs(y).max() + 1e-8)
    return y[: int(MAX_SECONDS * TARGET_SR)]

def handcrafted(y):
    mf = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=40)
    mf = np.vstack([mf, librosa.feature.delta(mf)])
    lm = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_mels=64))
    return np.concatenate([mf.mean(1), mf.std(1)]), np.concatenate([lm.mean(1), lm.std(1)])

In [ ]:
from tqdm.auto import tqdm
tag = f"n{N_PER_CORPUS}_s{SEED}"
f_cache = OUT_DIR / "features" / f"handcrafted_{tag}.npz"
if f_cache.exists():
    z = np.load(f_cache); X_mfcc, X_mel = z["mfcc"], z["mel"]
else:
    mf, ml = [], []
    for p in tqdm(work.path, desc="MFCC/log-mel"):
        y = preprocess(p); a, b = handcrafted(y); mf.append(a); ml.append(b)
    X_mfcc, X_mel = np.array(mf), np.array(ml)
    np.savez(f_cache, mfcc=X_mfcc, mel=X_mel)
work.to_csv(OUT_DIR / "features" / f"work_meta_{tag}.csv", index=False)
X_mfcc.shape, X_mel.shape

In [ ]:
import torch
from transformers import AutoFeatureExtractor, AutoModel
device = "cuda" if torch.cuda.is_available() else "cpu"

def ssl_embeddings(model_name, paths):
    fe = AutoFeatureExtractor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name, output_hidden_states=True).to(device).eval()
    out = []
    with torch.no_grad():
        for p in tqdm(paths, desc=model_name.split("/")[-1]):
            y = preprocess(p)
            inp = fe(y, sampling_rate=TARGET_SR, return_tensors="pt").to(device)
            hs = model(**inp).hidden_states                      # tuple(L+1) of [1, T, 768]
            out.append(torch.stack([h.mean(1)[0] for h in hs]).cpu().numpy())   # [L+1, 768]
    return np.stack(out)                                         # [N, L+1, 768]

EMB = {}
for short, name in SSL_MODELS.items():
    cache = OUT_DIR / "features" / f"{short}_{tag}.npy"
    EMB[short] = np.load(cache) if cache.exists() else ssl_embeddings(name, work.path.tolist())
    np.save(cache, EMB[short]); print(short, EMB[short].shape)

## 5. Quick model tests

All splits are **speaker-independent** (no speaker in both train and test) except the closed-set speaker-ID probe, which by definition needs the same speakers in train and test. Metrics: **UAR** (= balanced accuracy, key metric), macro-F1, and for attacks the **advantage over the majority-class baseline**.

In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score, mean_absolute_error

def clf(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5))

def eval_task(X, y, groups=None, test_size=0.25, speaker_independent=True):
    y = np.asarray(y)
    if speaker_independent:
        tr, te = next(GroupShuffleSplit(1, test_size=test_size, random_state=SEED).split(X, y, groups))
    else:
        tr, te = next(StratifiedShuffleSplit(1, test_size=test_size, random_state=SEED).split(X, y))
    m = clf().fit(X[tr], y[tr]); p = m.predict(X[te])
    maj = pd.Series(y[tr]).mode()[0]
    return dict(UAR=balanced_accuracy_score(y[te], p), F1=f1_score(y[te], p, average="macro"),
                acc=accuracy_score(y[te], p), majority_acc=accuracy_score(y[te], np.full(len(te), maj)))

LAYER = -1   # last layer; see the layer sweep below
FEATS = {"MFCC": X_mfcc, "log-mel": X_mel, **{k: v[:, LAYER] for k, v in EMB.items()}}

In [ ]:
# 5.1 SER baseline, per corpus, speaker-independent
res = []
for c, g in work.groupby("corpus"):
    idx = g.index.values
    if g.speaker.nunique() < 4: continue
    for fname, X in FEATS.items():
        r = eval_task(X[idx], g.emotion.values, g.speaker.values); r.update(corpus=c, features=fname, task="emotion"); res.append(r)
ser = pd.DataFrame(res).pivot(index="corpus", columns="features", values="UAR").round(3)
ser

In [ ]:
# 5.2 Attribute probes = first leakage check (what an attacker could recover from representations)
res = []
for fname, X in FEATS.items():
    mixed = work.groupby("corpus").gender.transform("nunique") > 1          # SAVEE is all male: exclude from gender
    i = work.index[mixed].values
    r = eval_task(X[i], work.gender.values[i], work.speaker.values[i]); r.update(features=fname, task="gender (unseen speakers)"); res.append(r)
    # closed-set speaker ID within each corpus, averaged
    ok = [g for c, g in work.groupby("corpus") if g.speaker.value_counts().min() >= 4]   # needs >= 4 clips per speaker
    sp = [eval_task(X[g.index.values], g.speaker.values, speaker_independent=False)["acc"] for g in ok]
    res.append(dict(features=fname, task="speaker ID (closed set, mean acc)", acc=np.mean(sp),
                    majority_acc=np.mean([1 / g.speaker.nunique() for g in ok])))
probe = pd.DataFrame(res); probe["advantage"] = probe.acc - probe.majority_acc
probe[["task","features","acc","majority_acc","advantage","UAR"]].round(3)

In [ ]:
# 5.3 Age probe (CREMA-D only has a wide age range): regression MAE on unseen speakers
g = work[work.corpus == "CREMA-D"]
if len(g):
    tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=SEED).split(g, groups=g.speaker))
    for fname, X in FEATS.items():
        Xi = X[g.index.values]; yv = g.age.values
        m = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(Xi[tr], yv[tr])
        print(f"{fname:9s} age MAE = {mean_absolute_error(yv[te], m.predict(Xi[te])):.1f} y   (predict-mean MAE = {mean_absolute_error(yv[te], np.full(len(te), yv[tr].mean())):.1f})")

In [ ]:
# 5.4 Layer sweep: which transformer layers carry emotion vs gender information
fig, ax = plt.subplots(1, len(EMB), figsize=(6*len(EMB), 4), squeeze=False)
mixed = (work.groupby("corpus").gender.transform("nunique") > 1).values
for k, (short, E) in enumerate(EMB.items()):
    emo, gen = [], []
    for L in range(E.shape[1]):
        emo.append(np.mean([eval_task(E[g.index.values, L], g.emotion.values, g.speaker.values)["UAR"]
                            for c, g in work.groupby("corpus") if g.speaker.nunique() >= 4]))
        gen.append(eval_task(E[mixed, L], work.gender.values[mixed], work.speaker.values[mixed])["UAR"])
    ax[0, k].plot(emo, "o-", label="emotion UAR"); ax[0, k].plot(gen, "s-", label="gender UAR (attack)")
    ax[0, k].set_title(short); ax[0, k].set_xlabel("layer"); ax[0, k].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / "layer_sweep.png", dpi=150); plt.show()

### 5.5 Minimal FedAvg with speaker-partitioned clients

Each **client = one speaker** (the realistic and worst case for privacy). Clients train a small MLP emotion head on frozen SSL embeddings; the server averages updates (FedAvg). Evaluation is on held-out speakers.

The honest-but-curious **server attack**: in every round the server keeps each client's update vector Δw and trains a classifier to predict the client's **gender** from Δw (leave-clients-out cross-validation). Set `DP_SIGMA > 0` to clip updates and add Gaussian noise (a simple DP-FedAvg proxy, no formal ε accounting here) and watch the privacy-utility trade-off.

In [ ]:
import torch.nn as nn
from sklearn.model_selection import cross_val_predict, GroupKFold

FL_CORPUS = "CREMA-D"        # many speakers, both genders; RAVDESS or IEMOCAP also work
FL_FEAT = "wav2vec2"
ROUNDS, LOCAL_EPOCHS, LR, CLIENT_FRAC = 30, 2, 1e-2, 0.5
CLIP, DP_SIGMA = 1.0, 0.0    # try DP_SIGMA = 0.5, 1.0

def run_fedavg(dp_sigma=DP_SIGMA, seed=SEED):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    g = work[work.corpus == FL_CORPUS].reset_index()
    X = torch.tensor(EMB[FL_FEAT][g["index"].values, LAYER], dtype=torch.float32)
    X = (X - X.mean(0)) / (X.std(0) + 1e-6)
    classes = sorted(g.emotion.unique()); y = torch.tensor(g.emotion.map({c: i for i, c in enumerate(classes)}).values)
    spk = g.speaker.unique(); rng.shuffle(spk)
    test_spk = set(spk[: max(2, len(spk) // 5)]); train_spk = [s for s in spk if s not in test_spk]
    te = g.speaker.isin(test_spk).values
    model = nn.Sequential(nn.Linear(X.shape[1], 128), nn.ReLU(), nn.Linear(128, len(classes)))
    flat = lambda m: torch.cat([p.data.view(-1) for p in m.parameters()])
    updates, upd_gender, upd_client, hist = [], [], [], []
    for rnd in range(ROUNDS):
        chosen = rng.choice(train_spk, max(2, int(CLIENT_FRAC * len(train_spk))), replace=False)
        w0 = flat(model).clone(); deltas, sizes = [], []
        for s in chosen:
            idx = np.where((g.speaker == s).values)[0]
            local = nn.Sequential(nn.Linear(X.shape[1], 128), nn.ReLU(), nn.Linear(128, len(classes)))
            local.load_state_dict(model.state_dict())
            opt = torch.optim.SGD(local.parameters(), lr=LR)
            for _ in range(LOCAL_EPOCHS):
                opt.zero_grad(); nn.functional.cross_entropy(local(X[idx]), y[idx]).backward(); opt.step()
            d = flat(local) - w0
            if dp_sigma > 0:
                d = d * min(1.0, CLIP / (d.norm().item() + 1e-12)) + torch.randn_like(d) * dp_sigma * CLIP
            deltas.append(d); sizes.append(len(idx))
            n_head = 128 * len(classes) + len(classes)       # attacker looks at the output-layer part of the update (small, fast)
            updates.append(d[-n_head:].numpy()); upd_gender.append(g.gender[idx[0]]); upd_client.append(s)
        avg = sum(d * n for d, n in zip(deltas, sizes)) / sum(sizes) if dp_sigma == 0 else torch.stack(deltas).mean(0)
        torch.nn.utils.vector_to_parameters(w0 + avg, model.parameters())
        with torch.no_grad(): p = model(X[te]).argmax(1).numpy()
        hist.append(balanced_accuracy_score(y[te].numpy(), p))
    # server-side gender inference on the stored updates, splitting by client so the attacker generalises to new clients
    U = np.array(updates); G = np.array(upd_gender)
    pred = cross_val_predict(clf(), U, G, groups=np.array(upd_client), cv=GroupKFold(5))
    return hist, balanced_accuracy_score(G, pred)

for sigma in [0.0, 0.5, 1.0]:
    hist, att = run_fedavg(sigma)
    plt.plot(hist, label=f"σ={sigma}: final UAR {hist[-1]:.2f}, gender attack UAR {att:.2f}")
    print(f"DP σ={sigma}: emotion UAR (held-out speakers) = {hist[-1]:.3f} | server gender-inference UAR = {att:.3f} (chance 0.5)")
plt.xlabel("round"); plt.ylabel("emotion UAR"); plt.title(f"FedAvg on {FL_CORPUS}, 1 speaker per client"); plt.legend()
plt.savefig(OUT_DIR / "figures" / "fedavg_dp.png", dpi=150); plt.show()

## 6. Save results and next steps

In [ ]:
ser.to_csv(OUT_DIR / f"ser_baseline_{tag}.csv"); probe.to_csv(OUT_DIR / f"attribute_probes_{tag}.csv", index=False)
print("Saved to", OUT_DIR)

**Reading the results against the proposed gates**: a defence passes if attacker advantage stays within **5 pp** of the majority baseline while emotion UAR drops by at most **5 points** (pending supervisor confirmation).

Suggested next steps:
* Scale `N_PER_CORPUS` up (or `None`) once everything runs; cached features are reused.
* Add gradient reversal on gender as an adversarial defence, and LoRA/adapters on the SSL encoder for the PEFT arm.
* Replace the DP proxy with Opacus or a proper RDP accountant to report ε = 1, 3, 5, 8.
* Add MSP-IMPROV once its remaining sessions and labels are available, and the standard Common Voice release for age/gender at scale.